# Module SC12 — Train `convnext_small.fb_in22k_ft_in1k` (single shared backbone + 4 heads)

> **Run on Kaggle** — Settings: Accelerator **GPU T4 x2**, Internet **ON**, Environment **Always use latest**. Attach inputs listed below, then **Save Version (Save & Run All)** so the outputs can be chained into the next module.

**Inputs:** `thesis-common-10c`, the image dataset, and Module-SC1 output (`config.json`).

**Outputs:** `logits_convnext_small.fb_in22k_ft_in1k.npz`, `weights_convnext_small.fb_in22k_ft_in1k.pt`, `metrics_convnext_small.fb_in22k_ft_in1k.json`, `history_convnext_small.fb_in22k_ft_in1k.csv` — attach to Modules SC6 & SC7.

In [ ]:

import sys, glob, os, json
import numpy as np
# Provenance bootstrap: explicit content identity; no timestamp/size selection.
import hashlib
from pathlib import Path
_PINNED_HELPER_SHA256 = "57b5c16cc9a08c7a3b2afa02c90ece96859b207d4533c78657f862752b68c34b"
ARTIFACT_OVERRIDES = {}  # T.find_one-only paths; generic cached artifact conflicts still require detaching.
_local_input_root = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
_input_roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [_local_input_root]
def _content_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def _input_candidates(pattern):
    hits = set()
    for root in _input_roots:
        for path in root.rglob(pattern):
            if path.is_file() and not {'Archives', '.git', 'node_modules'}.intersection(path.parts):
                hits.add(str(path.resolve()))
    return sorted(hits)
_helpers = _input_candidates('thesis_common.py')
_matching_helpers = [p for p in _helpers if _content_sha256(p) == _PINNED_HELPER_SHA256]
assert _matching_helpers, ('Attach the thesis_common.py supplied in Notebooks. '
                           'No input helper matches SHA256 ' + _PINNED_HELPER_SHA256)
_helper_path = _matching_helpers[0]
sys.path.insert(0, str(Path(_helper_path).parent))
import importlib, thesis_common as T
importlib.reload(T)
assert _content_sha256(T.__file__) == _PINNED_HELPER_SHA256, 'Wrong thesis_common imported.'
def _strict_find_one(filename_glob, required=True):
    if filename_glob in ARTIFACT_OVERRIDES:
        path = Path(ARTIFACT_OVERRIDES[filename_glob])
        assert path.is_file(), 'ARTIFACT_OVERRIDES path missing: ' + str(path)
        return str(path)
    hits = _input_candidates(filename_glob)
    if not hits:
        if required:
            raise FileNotFoundError('Attach the exact producer for ' + filename_glob + '; see the notebook preamble.')
        return None
    if len(hits) > 1:
        digests = {_content_sha256(p) for p in hits}
        if len(digests) != 1:
            raise RuntimeError('Different artifacts share ' + filename_glob + ':\n' + '\n'.join(hits)
                               + '\nDetach stale producers or set ARTIFACT_OVERRIDES explicitly.')
    return hits[0]
T.find_one = _strict_find_one
# Original local find()/find_all() consumers cannot obey T.find_one overrides.
# Reject conflicting standard cached artifacts up front instead of letting those
# consumers silently choose a same-name SC7/SC26 or stale output.
_generic_patterns = ('logits_*.npz', 'metrics_*.json', 'history_*.csv', 'calibration.json', 'config.json')
_by_basename = {}
for _pattern in _generic_patterns:
    for _p in _input_candidates(_pattern):
        _by_basename.setdefault(Path(_p).name, []).append(_p)
for _name, _paths in _by_basename.items():
    if len(_paths) > 1 and len({_content_sha256(p) for p in _paths}) > 1:
        raise RuntimeError('Conflicting input producers for ' + _name + ':\n' + '\n'.join(_paths)
                           + '\nAttach one producer/version per basename for this run.')
print('Pinned helper:', _helper_path, '| SHA256:', _PINNED_HELPER_SHA256)
_cfg = T.find_one('config.json', required=False)
if _cfg: T.load_config(_cfg); print('loaded config:', _cfg)
print('device:', T.DEVICE, '| materials:', len(T.MATERIALS), '| epochs:', T.EPOCHS)


In [ ]:
BACKBONE = "convnext_small.fb_in22k_ft_in1k"
print("training:", BACKBONE)

In [ ]:

import torch
ROOT = T.find_image_root()
df_tr, df_va, df_te = T.load_manifests(ROOT); T.check_leakage(df_tr, df_va, df_te)
model, hist = T.train_backbone(BACKBONE, df_tr, df_va, df_te)

### Evaluate on val+test, cache logits, save weights/history

In [ ]:

_, va, te = T.make_loaders(df_tr, df_va, df_te)
val  = T.evaluate_logits(model, va)
test = T.evaluate_logits(model, te)
T.save_logits(f'logits_{BACKBONE}.npz', val, test)
torch.save(model.state_dict(), f'weights_{BACKBONE}.pt')
hist.to_csv(f'history_{BACKBONE}.csv', index=False)
print('saved logits/weights/history for', BACKBONE)

### Rich test metrics — per-class reports + confusion + threat-by-level (find the weak heads)

In [ ]:

rep = T.full_report(*test)
json.dump(rep, open(f'metrics_{BACKBONE}.json','w'), indent=2)
s = rep['scalars']
print('SCALARS  material:', s['material'], '\n         route   :', s['route'],
      '\n         threat  :', s['threat'], '\n         condition:', s['condition'])
print('\nMATERIAL per-class F1 (worst first):')
for k,v in sorted(rep['per_class']['material'].items(), key=lambda kv: kv[1]['f1']):
    print(f"   {k:20s} f1={v['f1']:.2f}  P={v['precision']:.2f} R={v['recall']:.2f}  n={v['support']}")
print('\nROUTE per-class:')
for k,v in rep['per_class']['route'].items():
    print(f"   {k:10s} f1={v['f1']:.2f}  n={v['support']}")
print('\nTHREAT per level acc:', rep['threat_per_level'])
print('\nROUTE confusion (rows=true):'); print(np.array(rep['confusion']['route']))

### Training curves

In [ ]:

import matplotlib.pyplot as plt
h = hist
fig, ax = plt.subplots(1,2, figsize=(11,3.5))
ax[0].plot(h.epoch, h.train_loss, marker='o', label='train')
if 'val_loss' in h.columns: ax[0].plot(h.epoch, h.val_loss, marker='s', label='val')
ax[0].legend(); ax[0].set_title('loss: train vs val (overfitting gap)'); ax[0].set_xlabel('epoch')
for col in ['val_route_f1','val_threat_qwk','val_material_f1','val_condition_f1']:
    ax[1].plot(h.epoch, h[col], marker='.', label=col)
ax[1].legend(); ax[1].set_title('val metrics'); ax[1].set_xlabel('epoch'); plt.tight_layout(); plt.show()
print('best val_loss epoch:', int(h.loc[h.val_loss.idxmin(),'epoch']) if 'val_loss' in h.columns else 'n/a',
      '| final train/val loss: %.3f / %.3f' % (h.train_loss.iloc[-1], h.val_loss.iloc[-1] if 'val_loss' in h.columns else float('nan')))